# Notebook 3: K-Means Clustering — Customer Segmentation
## ST7082CEM — Big Data Management and Data Visualisation
### Smaran Luitel | Student ID: 250087

This notebook applies K-Means clustering to discover natural customer segments independent of the churn label.

**Justification:** Clustering is an unsupervised technique used to identify homogeneous groups of customers based on financial and behavioural attributes. These segments can inform targeted retention strategies. Unlike classification, clustering requires no labelled data and reveals structure the business may not have predefined.

The elbow method (WSSSE) and silhouette score are used to determine the optimal number of clusters.

In [1]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = "C:/Users/user/AppData/Local/Programs/Python/Python314/python.exe"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ST7082CEM_Clustering_250087") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


## 1. Load Prepared Data

In [2]:
# Load raw CSV and apply same column renames as preprocessing notebook
df_prepared = (
    spark.read.csv("../dataset/Customer-Churn-Records.csv", header=True, inferSchema=True)
    .withColumnRenamed("Satisfaction Score", "SatisfactionScore")
    .withColumnRenamed("Card Type", "CardType")
    .withColumnRenamed("Point Earned", "PointEarned")
)
print(f"Loaded {df_prepared.count()} rows")

Loaded 10000 rows


## 2. Prepare Clustering Features

A clustering-specific feature vector is assembled from the most meaningful financial and behavioural dimensions. The churn label (`Exited`) is deliberately excluded — clustering is unsupervised and should not be guided by the outcome.

**Selected features:** CreditScore, Age, Balance, NumOfProducts, EstimatedSalary, SatisfactionScore, PointEarned

In [3]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

cluster_cols = ["CreditScore", "Age", "Balance", "NumOfProducts",
                "EstimatedSalary", "SatisfactionScore", "PointEarned"]

assembler_clust = VectorAssembler(
    inputCols=cluster_cols,
    outputCol="cluster_features_raw",
    handleInvalid="keep"
)
scaler_clust = StandardScaler(
    inputCol="cluster_features_raw",
    outputCol="cluster_features",
    withMean=True,
    withStd=True
)

clust_pipeline = Pipeline(stages=[assembler_clust, scaler_clust])
df_clust = clust_pipeline.fit(df_prepared).transform(df_prepared)

print(f"Clustering features assembled: {cluster_cols}")
df_clust.select("cluster_features").show(3, truncate=False)

Clustering features assembled: ['CreditScore', 'Age', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'SatisfactionScore', 'PointEarned']


+----------------------------------------------------------------------------------------------------------------------------------------------+
|cluster_features                                                                                                                              |
+----------------------------------------------------------------------------------------------------------------------------------------------+
|[-0.3262051105578709,0.2935027466586979,-1.2257863774930702,-0.9115379137259417,0.021885399643330136,-0.7210943589200214,-0.6308075751822447] |
|[-0.44001395250987396,0.19815392375612903,0.11734415378734744,-0.9115379137259417,0.2165229249290683,-0.00981564623504683,-0.6662175815707467]|
|[-1.536717338592813,0.2935027466586979,1.3329866913892099,2.526930263286268,0.2406748654176144,-0.00981564623504683,-1.0158913946572041]      |
+---------------------------------------------------------------------------------------------------------------------------------

## 3. Elbow Method — Optimal k Selection

K-Means requires the number of clusters `k` to be specified in advance. Two metrics guide this choice:
- **WSSSE** (Within-Cluster Sum of Squared Errors): decreases as k increases; look for the "elbow" where the gain diminishes
- **Silhouette Score**: measures how similar points are to their own cluster vs others; higher is better (range -1 to 1)

In [4]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import pandas as pd

evaluator = ClusteringEvaluator(featuresCol="cluster_features", metricName="silhouette")

costs = []
silhouettes = []
k_range = range(2, 9)

for k in k_range:
    km = KMeans(k=k, featuresCol="cluster_features", seed=42, maxIter=100)
    km_model = km.fit(df_clust)
    predictions = km_model.transform(df_clust)
    wssse = km_model.summary.trainingCost
    sil = evaluator.evaluate(predictions)
    costs.append(wssse)
    silhouettes.append(sil)
    print(f"k={k} | WSSSE: {wssse:>12.2f} | Silhouette: {sil:.4f}")

elbow_df = pd.DataFrame({"k": list(k_range), "WSSSE": costs, "Silhouette": silhouettes})
elbow_df.to_csv("../exports/kmeans_elbow.csv", index=False)
print("\nSaved: ../exports/kmeans_elbow.csv")

k=2 | WSSSE:     59849.45 | Silhouette: 0.2586


k=3 | WSSSE:     55196.27 | Silhouette: 0.2054


k=4 | WSSSE:     51882.22 | Silhouette: 0.1901


k=5 | WSSSE:     48222.29 | Silhouette: 0.2067


k=6 | WSSSE:     45711.01 | Silhouette: 0.1986


k=7 | WSSSE:     43817.14 | Silhouette: 0.1994


k=8 | WSSSE:     42032.15 | Silhouette: 0.1980

Saved: ../exports/kmeans_elbow.csv


## 4. Fit Final K-Means Model

Select the optimal `k` based on the elbow chart above. Update `OPTIMAL_K` accordingly before proceeding.

In [5]:
# Update this value based on the elbow chart output above
OPTIMAL_K = 4

final_km = KMeans(k=OPTIMAL_K, featuresCol="cluster_features", seed=42, maxIter=100)
final_km_model = final_km.fit(df_clust)
df_clustered = final_km_model.transform(df_clust)

print(f"Final model: K-Means with k={OPTIMAL_K}")
print(f"Training WSSSE: {final_km_model.summary.trainingCost:.2f}")

print("\nCluster size distribution:")
df_clustered.groupBy("prediction").count() \
    .withColumn("pct", F.round(F.col("count") / df_clustered.count() * 100, 2)) \
    .orderBy("prediction").show()

Final model: K-Means with k=4
Training WSSSE: 51882.22

Cluster size distribution:


+----------+-----+-----+
|prediction|count|  pct|
+----------+-----+-----+
|         0| 2112|21.12|
|         1| 3095|30.95|
|         2| 2529|25.29|
|         3| 2264|22.64|
+----------+-----+-----+



## 5. Cluster Profiling

Profile each cluster across key dimensions to give them business-meaningful interpretations.

In [6]:
profile_aggs = [
    F.count("*").alias("size"),
    F.round(F.mean("CreditScore"), 1).alias("avg_CreditScore"),
    F.round(F.mean("Age"), 1).alias("avg_Age"),
    F.round(F.mean("Balance"), 1).alias("avg_Balance"),
    F.round(F.mean("NumOfProducts"), 2).alias("avg_NumOfProducts"),
    F.round(F.mean("EstimatedSalary"), 1).alias("avg_EstimatedSalary"),
    F.round(F.mean("SatisfactionScore"), 2).alias("avg_SatisfactionScore"),
    F.round(F.mean("PointEarned"), 1).alias("avg_PointEarned"),
    F.round(F.mean("Exited") * 100, 2).alias("churn_rate_pct"),
]

cluster_profile = df_clustered.groupBy("prediction").agg(*profile_aggs).orderBy("prediction")

print("Cluster Profiles:")
cluster_profile.show(truncate=False)

Cluster Profiles:


+----------+----+---------------+-------+-----------+-----------------+-------------------+---------------------+---------------+--------------+
|prediction|size|avg_CreditScore|avg_Age|avg_Balance|avg_NumOfProducts|avg_EstimatedSalary|avg_SatisfactionScore|avg_PointEarned|churn_rate_pct|
+----------+----+---------------+-------+-----------+-----------------+-------------------+---------------------+---------------+--------------+
|0         |2112|640.7          |39.4   |103456.0   |1.26             |95604.9            |4.08                 |410.9          |23.63         |
|1         |3095|651.6          |38.0   |10884.1    |2.1              |99307.4            |3.0                  |602.5          |12.37         |
|2         |2529|654.1          |39.1   |106469.8   |1.29             |102476.0           |1.47                 |577.4          |24.63         |
|3         |2264|654.3          |39.6   |107513.8   |1.27             |102679.5           |3.77                 |827.0          |2

## 6. Cluster Churn Analysis

In [7]:
print("Churn rate by cluster:")
df_clustered.groupBy("prediction", "Exited").count() \
    .orderBy("prediction", "Exited").show()

print("Geography mix by cluster:")
df_clustered.groupBy("prediction", "Geography").count() \
    .orderBy("prediction", "count", ascending=[True, False]).show()

print("Gender mix by cluster:")
df_clustered.groupBy("prediction", "Gender").count() \
    .orderBy("prediction", "count", ascending=[True, False]).show()

Churn rate by cluster:


+----------+------+-----+
|prediction|Exited|count|
+----------+------+-----+
|         0|     0| 1613|
|         0|     1|  499|
|         1|     0| 2712|
|         1|     1|  383|
|         2|     0| 1906|
|         2|     1|  623|
|         3|     0| 1731|
|         3|     1|  533|
+----------+------+-----+

Geography mix by cluster:


+----------+---------+-----+
|prediction|Geography|count|
+----------+---------+-----+
|         0|   France|  957|
|         0|  Germany|  685|
|         0|    Spain|  470|
|         1|   France| 1943|
|         1|    Spain|  964|
|         1|  Germany|  188|
|         2|   France| 1101|
|         2|  Germany|  866|
|         2|    Spain|  562|
|         3|   France| 1013|
|         3|  Germany|  770|
|         3|    Spain|  481|
+----------+---------+-----+

Gender mix by cluster:
+----------+------+-----+
|prediction|Gender|count|
+----------+------+-----+
|         0|  Male| 1160|
|         0|Female|  952|
|         1|  Male| 1683|
|         1|Female| 1412|
|         2|  Male| 1391|
|         2|Female| 1138|
|         3|  Male| 1223|
|         3|Female| 1041|
+----------+------+-----+



## 7. Export Cluster Assignments for Tableau

In [8]:
from pyspark.sql.window import Window

# Add a row index to both DataFrames for reliable join
w = Window.orderBy(F.monotonically_increasing_id())

# Re-load the original CSV to get raw readable columns for Tableau
df_original = spark.read.csv("../dataset/Customer-Churn-Records.csv", header=True, inferSchema=True) \
    .withColumnRenamed("Satisfaction Score", "SatisfactionScore") \
    .withColumnRenamed("Card Type", "CardType") \
    .withColumnRenamed("Point Earned", "PointEarned")

# Extract cluster labels aligned by monotonic id
df_original_idx = df_original.withColumn("_row_id", F.monotonically_increasing_id())
df_cluster_idx  = df_clustered.select("prediction").withColumn("_row_id", F.monotonically_increasing_id())

df_with_cluster = df_original_idx.join(df_cluster_idx, on="_row_id", how="left") \
    .drop("_row_id") \
    .withColumnRenamed("prediction", "Cluster")

df_with_cluster.toPandas().to_csv("../exports/churn_with_clusters.csv", index=False)
print(f"Saved: ../exports/churn_with_clusters.csv ({df_with_cluster.count()} rows)")

Saved: ../exports/churn_with_clusters.csv (10000 rows)
